# Postgres Data Explorer

Use this notebook to inspect the Azure Postgres data behind the dataload and processing pipeline. It covers raw source records, normalized metadata, enrichment outputs, embeddings metadata, semantic connections, ticker summaries, sentiment aggregates, and run history.

The notebook does not write data. It only reads from Postgres.

## 1. Setup

Connection options:

- Preferred: set `AZURE_POSTGRES_DSN` in your environment.
- Convenience: keep Azure CLI logged in and let the notebook fetch `azure-postgres-dsn` from Key Vault.

If your notebook kernel is missing packages, run the install cell once.

In [ ]:
# Uncomment if your notebook environment does not already have these packages.
# %pip install pandas psycopg[binary]

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path
from typing import Any

import pandas as pd
import psycopg

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 240)
pd.set_option("display.width", 180)

KEY_VAULT_NAME = "elizaaiworkben-kv-58415"
POSTGRES_SECRET_NAME = "azure-postgres-dsn"
AZ_CANDIDATES = [
    "az",
    r"C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin\az.cmd",
]


def _fetch_secret_with_az(vault_name: str, secret_name: str) -> str | None:
    for az_cmd in AZ_CANDIDATES:
        try:
            value = subprocess.check_output(
                [
                    az_cmd,
                    "keyvault",
                    "secret",
                    "show",
                    "--vault-name",
                    vault_name,
                    "--name",
                    secret_name,
                    "--query",
                    "value",
                    "-o",
                    "tsv",
                ],
                text=True,
                stderr=subprocess.DEVNULL,
            ).strip()
            if value:
                return value
        except Exception:
            continue
    return None


def get_postgres_dsn() -> str:
    dsn = os.getenv("AZURE_POSTGRES_DSN")
    if dsn:
        return dsn
    dsn = _fetch_secret_with_az(KEY_VAULT_NAME, POSTGRES_SECRET_NAME)
    if dsn:
        return dsn
    raise RuntimeError(
        "Set AZURE_POSTGRES_DSN or login with Azure CLI so the notebook can read the Key Vault secret."
    )


DSN = get_postgres_dsn()
conn = psycopg.connect(DSN)
print("Connected to Postgres. DSN loaded without printing credentials.")

## 2. Query Helpers

These helpers keep result sets readable and preserve JSON metadata columns as Python objects where possible.

In [ ]:
def q(sql: str, params: tuple[Any, ...] | dict[str, Any] | None = None) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn, params=params)


def table_counts() -> pd.DataFrame:
    tables = [
        "source_items",
        "dataload_runs",
        "processing_runs",
        "item_enrichments",
        "item_embeddings",
        "item_connections",
        "brain_summaries",
        "sentiment_weekly",
    ]
    rows = []
    with conn.cursor() as cur:
        for table in tables:
            try:
                cur.execute(f"select count(*) from {table}")
                rows.append({"table": table, "rows": cur.fetchone()[0]})
            except Exception as exc:
                conn.rollback()
                rows.append({"table": table, "rows": None, "error": str(exc)[:180]})
    return pd.DataFrame(rows)


def table_schema(table_name: str) -> pd.DataFrame:
    return q(
        """
        select ordinal_position, column_name, data_type, udt_name, is_nullable, column_default
        from information_schema.columns
        where table_schema = 'public' and table_name = %s
        order by ordinal_position
        """,
        (table_name,),
    )


def table_indexes(table_name: str) -> pd.DataFrame:
    return q(
        """
        select indexname, indexdef
        from pg_indexes
        where schemaname = 'public' and tablename = %s
        order by indexname
        """,
        (table_name,),
    )


def sample_table(table_name: str, limit: int = 25) -> pd.DataFrame:
    if not table_name.replace("_", "").isalnum():
        raise ValueError("Unsafe table name")
    return q(f"select * from {table_name} limit %s", (limit,))

## 3. Database Inventory

In [ ]:
table_counts()

In [ ]:
q(
    """
    select table_name
    from information_schema.tables
    where table_schema = 'public'
    order by table_name
    """
)

In [ ]:
# Change this table name to inspect columns and indexes.
TABLE_TO_INSPECT = "source_items"
display(table_schema(TABLE_TO_INSPECT))
display(table_indexes(TABLE_TO_INSPECT))

## 4. Source Data Overview

`source_items` is the normalized source table. Raw payloads are in Azure Blob; normalized titles, bodies, URLs, authors, timestamps, and source metadata live here.

In [ ]:
q(
    """
    select ticker, source,
           count(*) as item_count,
           min(published_at) as earliest_published_at,
           max(published_at) as latest_published_at,
           max(fetched_at) as latest_fetched_at
    from source_items
    group by ticker, source
    order by ticker, source
    """
)

In [ ]:
def show_source_items(ticker: str | None = None, source: str | None = None, limit: int = 50) -> pd.DataFrame:
    filters = []
    params: list[Any] = []
    if ticker:
        filters.append("ticker = %s")
        params.append(ticker)
    if source:
        filters.append("source = %s")
        params.append(source)
    where = "where " + " and ".join(filters) if filters else ""
    params.append(limit)
    return q(
        f"""
        select source_item_id, ticker, source, published_at, fetched_at,
               title, left(body, 700) as body_preview, source_url, author, metadata
        from source_items
        {where}
        order by published_at desc nulls last, fetched_at desc
        limit %s
        """,
        tuple(params),
    )


# Examples: show_source_items("AMD", "reddit"), show_source_items("SNDK"), show_source_items(source="hacker_news")
show_source_items("AMD", limit=20)

## 5. Source Metadata By Source

This helps compare which metadata fields each source is contributing.

In [ ]:
q(
    """
    select source,
           array_agg(distinct key order by key) as metadata_keys
    from source_items,
         lateral jsonb_object_keys(metadata) as key
    group by source
    order by source
    """
)

In [ ]:
def show_metadata_samples(source: str, limit: int = 10) -> pd.DataFrame:
    return q(
        """
        select source_item_id, ticker, source, title, metadata
        from source_items
        where source = %s
        order by fetched_at desc
        limit %s
        """,
        (source, limit),
    )


show_metadata_samples("github")

## 6. Dataload Runs

Use this to understand freshness, incremental windows, and source/ticker load health.

In [ ]:
q(
    """
    select source, ticker, status, count(*) as run_count,
           max(started_at) as latest_started_at,
           max(completed_at) as latest_completed_at,
           max(window_end) as latest_window_end
    from dataload_runs
    group by source, ticker, status
    order by source, ticker, status
    """
)

In [ ]:
q(
    """
    select run_id, source, ticker, status, started_at, completed_at,
           window_start, window_end, raw_blob_path, normalized_blob_path,
           item_count, error_message, metadata
    from dataload_runs
    order by started_at desc
    limit 50
    """
)

## 7. Processing Coverage

This mirrors the deployed `processing_status` endpoint: it shows source records, enriched items, embedded items, summaries, connections, and sentiment rows per ticker.

In [ ]:
q(
    """
    with source_counts as (
        select ticker, count(*) as source_items from source_items group by ticker
    ), enrich_counts as (
        select ticker, count(*) as enriched_items,
               count(*) filter (where relevance >= 0) as relevant_items,
               max(relevance) as max_relevance
        from item_enrichments group by ticker
    ), embedding_counts as (
        select ticker, count(*) as embedded_items from item_embeddings group by ticker
    ), connection_counts as (
        select ticker, count(*) as connections_total,
               count(*) filter (where valid = true) as valid_connections
        from item_connections group by ticker
    ), summary_counts as (
        select ticker, count(*) as summaries from brain_summaries group by ticker
    ), sentiment_counts as (
        select ticker, count(*) as sentiment_rows from sentiment_weekly group by ticker
    )
    select sc.ticker,
           sc.source_items,
           coalesce(ec.enriched_items, 0) as enriched_items,
           coalesce(ec.relevant_items, 0) as relevant_items,
           coalesce(emc.embedded_items, 0) as embedded_items,
           coalesce(cc.connections_total, 0) as connections_total,
           coalesce(cc.valid_connections, 0) as valid_connections,
           coalesce(suc.summaries, 0) as summaries,
           coalesce(sec.sentiment_rows, 0) as sentiment_rows,
           ec.max_relevance
    from source_counts sc
    left join enrich_counts ec using (ticker)
    left join embedding_counts emc using (ticker)
    left join connection_counts cc using (ticker)
    left join summary_counts suc using (ticker)
    left join sentiment_counts sec using (ticker)
    order by sc.ticker
    """
)

## 8. Enriched Items

Enrichments contain LLM-scored relevance, sentiment, themes, firsthand flags, and analyst-oriented summaries.

In [ ]:
def show_enrichments(ticker: str | None = None, source: str | None = None, limit: int = 50) -> pd.DataFrame:
    filters = []
    params: list[Any] = []
    if ticker:
        filters.append("ie.ticker = %s")
        params.append(ticker)
    if source:
        filters.append("ie.source = %s")
        params.append(source)
    where = "where " + " and ".join(filters) if filters else ""
    params.append(limit)
    return q(
        f"""
        select ie.source_item_id, ie.ticker, ie.source, si.published_at,
               ie.relevance, ie.sentiment, ie.sentiment_rationale,
               ie.themes, ie.firsthand, ie.firsthand_type,
               ie.summary, si.title, si.source_url, si.metadata, ie.model, ie.enriched_at
        from item_enrichments ie
        join source_items si on si.source_item_id = ie.source_item_id
        {where}
        order by ie.relevance desc, si.published_at desc nulls last
        limit %s
        """,
        tuple(params),
    )


show_enrichments(limit=25)

## 9. Embeddings Metadata

This intentionally avoids displaying full vectors. It shows embedding coverage, model, dimensions, and linked source/enrichment metadata.

In [ ]:
q(
    """
    select emb.ticker, emb.source, emb.model,
           count(*) as embedded_items,
           min(emb.published_at) as earliest_published_at,
           max(emb.published_at) as latest_published_at,
           max(emb.embedded_at) as latest_embedded_at
    from item_embeddings emb
    group by emb.ticker, emb.source, emb.model
    order by emb.ticker, emb.source, emb.model
    """
)

In [ ]:
def show_embeddings(ticker: str | None = None, source: str | None = None, limit: int = 25) -> pd.DataFrame:
    filters = []
    params: list[Any] = []
    if ticker:
        filters.append("emb.ticker = %s")
        params.append(ticker)
    if source:
        filters.append("emb.source = %s")
        params.append(source)
    where = "where " + " and ".join(filters) if filters else ""
    params.append(limit)
    return q(
        f"""
        select emb.source_item_id, emb.ticker, emb.source, emb.published_at,
               emb.summary, emb.model, emb.embedded_at,
               vector_dims(emb.embedding) as embedding_dimensions,
               ie.relevance, ie.sentiment, ie.themes
        from item_embeddings emb
        join item_enrichments ie on ie.source_item_id = emb.source_item_id
        {where}
        order by emb.embedded_at desc
        limit %s
        """,
        tuple(params),
    )


show_embeddings(limit=25)

## 10. Semantic Connections

Connections are semantic review leads across sources for the same ticker.

In [ ]:
def show_connections(ticker: str | None = None, valid_only: bool = False, limit: int = 50) -> pd.DataFrame:
    filters = []
    params: list[Any] = []
    if ticker:
        filters.append("ic.ticker = %s")
        params.append(ticker)
    if valid_only:
        filters.append("ic.valid = true")
    where = "where " + " and ".join(filters) if filters else ""
    params.append(limit)
    return q(
        f"""
        select ic.connection_id, ic.ticker, ic.valid, ic.confidence, ic.connection_type,
               ic.source_a, ic.source_b, ic.item_a_id, ic.item_b_id,
               ic.similarity, ic.narrative, ic.stock_relevance,
               a.title as item_a_title, b.title as item_b_title,
               a.source_url as item_a_url, b.source_url as item_b_url,
               ic.model, ic.verified_at, ic.metadata
        from item_connections ic
        left join source_items a on a.source_item_id = ic.item_a_id
        left join source_items b on b.source_item_id = ic.item_b_id
        {where}
        order by ic.valid desc, ic.confidence desc, ic.verified_at desc
        limit %s
        """,
        tuple(params),
    )


show_connections(limit=25)

## 11. Brain Summaries

Ticker-level summaries include key signals, connection notes, bear case, confidence, cited item IDs, citation validation, search logs, and run/model metadata.

In [ ]:
def show_summaries(ticker: str | None = None, limit: int = 25) -> pd.DataFrame:
    filters = []
    params: list[Any] = []
    if ticker:
        filters.append("ticker = %s")
        params.append(ticker)
    where = "where " + " and ".join(filters) if filters else ""
    params.append(limit)
    return q(
        f"""
        select summary_id, ticker, headline, key_signals, cross_source_connections,
               bear_case, confidence, cited_item_ids, invalid_citation_ids,
               search_log, model, run_id, generated_at
        from brain_summaries
        {where}
        order by generated_at desc
        limit %s
        """,
        tuple(params),
    )


show_summaries(limit=10)

## 12. Weekly Sentiment

Weekly sentiment is aggregated from item enrichments by ticker and source.

In [ ]:
q(
    """
    select ticker, source,
           count(*) as week_rows,
           min(week_start) as first_week,
           max(week_start) as latest_week,
           sum(item_count) as total_items,
           avg(sentiment_avg) as avg_weekly_sentiment,
           count(*) filter (where alert = true) as alert_weeks
    from sentiment_weekly
    group by ticker, source
    order by ticker, source
    """
)

In [ ]:
def show_sentiment(ticker: str | None = None, source: str | None = None) -> pd.DataFrame:
    filters = []
    params: list[Any] = []
    if ticker:
        filters.append("ticker = %s")
        params.append(ticker)
    if source:
        filters.append("source = %s")
        params.append(source)
    where = "where " + " and ".join(filters) if filters else ""
    return q(
        f"""
        select ticker, source, week_start, item_count, sentiment_avg,
               rolling_mean_8w, rolling_stddev_8w, z_score, alert, refreshed_at
        from sentiment_weekly
        {where}
        order by ticker, source, week_start
        """,
        tuple(params),
    )


show_sentiment("AMD")

## 13. Processing Runs

In [ ]:
q(
    """
    select run_id, status, started_at, completed_at, error_message, metadata
    from processing_runs
    order by started_at desc
    limit 50
    """
)

## 14. Join A Source Item Through The Pipeline

Set `SOURCE_ITEM_ID` to trace one item from `source_items` into enrichment, embedding, and connection records.

In [ ]:
SOURCE_ITEM_ID = ""  # Example: "github:AMD:..."

if SOURCE_ITEM_ID:
    display(q("select * from source_items where source_item_id = %s", (SOURCE_ITEM_ID,)))
    display(q("select * from item_enrichments where source_item_id = %s", (SOURCE_ITEM_ID,)))
    display(
        q(
            """
            select source_item_id, ticker, source, published_at, summary, model,
                   embedded_at, vector_dims(embedding) as embedding_dimensions
            from item_embeddings
            where source_item_id = %s
            """,
            (SOURCE_ITEM_ID,),
        )
    )
    display(
        q(
            """
            select *
            from item_connections
            where item_a_id = %s or item_b_id = %s
            order by confidence desc
            """,
            (SOURCE_ITEM_ID, SOURCE_ITEM_ID),
        )
    )
else:
    print("Set SOURCE_ITEM_ID above to trace a specific item.")

## 15. Custom SQL Scratchpad

Edit `CUSTOM_SQL` for ad hoc inspection. Keep it read-only.

In [ ]:
CUSTOM_SQL = """
select ticker, source, count(*) as rows
from source_items
group by ticker, source
order by ticker, source
"""

q(CUSTOM_SQL)

## 16. Close Connection

In [ ]:
# Run when finished.
# conn.close()